# R(2+1)D-18 features + Attention MIL (whole video)

In `02-r2plus1d_fullvideo.ipynb` the features of **all** clips of a video were averaged into one vector. An anomaly usually
lasts a few seconds of a long video (e.g. 19 anomalous clips out of 2,800), so averaging dilutes it until the anomalous
video looks almost like a normal one.

This notebook treats each video as a **bag of clips** (Multiple Instance Learning). We only know the label of the whole
video, not which clips are anomalous:
- **anomaly video**: at least one part of it is anomalous;
- **normal video**: no part of it is anomalous.

A **gated attention** head (Ilse et al., 2018) learns a weight for every clip and classifies the attention-weighted
feature, so the few clips that contain the event can decide the prediction for the whole video. The attention weights also
show **where** in the video the anomaly is.

Pipeline:
1. **Feature extraction (once):** frozen Kinetics-400 R(2+1)D-18, every frame of every video, consecutive 16-frame clips
   (the same input as notebook 02) → one 512-d feature per clip, cached to disk. Two views: centre crop and its mirror image.
2. **Part A:** train the attention-MIL head on the *same* TRAIN/VAL split as notebook 02 → directly comparable numbers.
3. **Part B:** 5-fold cross-validation over all labelled videos (TRAIN + VAL). With only 53 VAL videos one video is
   ~1.9 % accuracy, so the cross-validated accuracy (350 videos) is the number to trust.
4. TEST predictions (Part A model and the 5-fold ensemble), anomaly timelines, comparison with notebook 02.

## 1. Configuration — change paths here

In [ ]:
import os

# ---------------- PATHS (edit these if you move the notebook / dataset) ----------------
DATASET_ROOT = "/home/haseebumer/content/data/UCF_Crimes_MINI"   # Videos/ + Anomaly_Detection_splits/
SPLITS_DIR   = os.path.join(DATASET_ROOT, "Anomaly_Detection_splits")
FEATS_DIR    = "./features_r2plus1d"     # cached clip features: <view>/<category>/<video>.npy, (N_clips, 512) float16
RUNS_DIR     = "./runs_ucf_attnmil"
PREV_RUNS_DIR = "./runs_ucf_fullvideo"   # notebook 02 (mean pooling), for the comparison table

# ---------------- SPLITS ----------------
VAL_FRACTION = 0.15        # same stratified hold-out as notebook 02 (same seed) -> Part A is directly comparable
N_FOLDS      = 5           # Part B: cross-validation over all labelled videos (TRAIN + VAL)

# ---------------- FEATURE EXTRACTION (frozen R(2+1)D-18, Kinetics-400) ----------------
# Every frame is used: a video is cut into consecutive, non-overlapping 16-frame clips (last one padded), as in notebook 02.
CLIP_LEN     = 16
RESIZE_HW    = (128, 171)  # resize (H, W) before cropping, as in torchvision's Kinetics recipe
CROP         = 112         # centre crop
CLIPS_PER_SEGMENT = 32     # a video is decoded and sent to the GPU 32 clips (512 frames) at a time
NUM_WORKERS  = 8           # CPU decode processes, 1 FFmpeg thread each
PREFETCH     = 2
VIEWS        = {"center": False, "center_flip": True}   # view name -> horizontal flip. Train: a random view per video
                                                        # per epoch (augmentation). Eval: logits averaged over all views.

# ---------------- ATTENTION-MIL HEAD ----------------
HIDDEN       = 256         # clip embedding size after the temporal conv
ATTN_DIM     = 128         # size of the gated attention layer
DROPOUT      = 0.5
BATCH_VIDEOS = 8           # videos per optimizer step (videos have different lengths -> gradient accumulation)
EPOCHS       = 40
LR           = 3e-4
WEIGHT_DECAY = 1e-2

SEED = 42

os.makedirs(RUNS_DIR, exist_ok=True)
assert os.path.isdir(DATASET_ROOT), f"DATASET_ROOT not found: {DATASET_ROOT}"
assert os.path.isdir(SPLITS_DIR), f"SPLITS_DIR not found: {SPLITS_DIR}"

## 2. Imports and GPU setup

In [ ]:
import random, time, copy, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm.auto import tqdm


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False                # the MIL head sees a different sequence length for every video;
                                                      # benchmark mode would re-tune the conv for each new length
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16

print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f"| free {free/2**30:.1f} GB / {total/2**30:.1f} GB")

## 3. Build the file lists (0 = NORMAL, 1 = ANOMALY; VAL is the same stratified hold-out as notebook 02; TEST has no labels)

In [ ]:
CLASSES = ["NORMAL", "ANOMALY"]
VIDEOS_DIR = os.path.join(DATASET_ROOT, "Videos")

def category(path):
    """Folder name of the video, e.g. 'Robbery' or 'Training_Normal_Videos_Anomaly'."""
    return os.path.basename(os.path.dirname(path))

def read_split(fname, labelled=True):
    """labelled=False: label is -1 and is never used (TEST is treated like unseen real-world data)."""
    items = []
    for line in open(os.path.join(SPLITS_DIR, fname)):
        rel = line.strip()
        if not rel:
            continue
        path = os.path.join(VIDEOS_DIR, rel)
        assert os.path.isfile(path), f"missing video: {path}"
        items.append((path, (0 if "Normal" in category(path) else 1) if labelled else -1))
    return items


train_all = read_split("Anomaly_Train.txt")
test_items = read_split("Anomaly_Test.txt", labelled=False)
train_items, val_items = train_test_split(train_all, test_size=VAL_FRACTION, random_state=SEED,
                                          stratify=[category(p) for p, _ in train_all])
train_items, val_items = sorted(train_items), sorted(val_items)

# All labelled videos in one list: Part A uses TRAIN_IDX / VAL_IDX, Part B splits all of them into folds.
LAB_ITEMS = train_items + val_items
TRAIN_IDX = np.arange(len(train_items))
VAL_IDX   = np.arange(len(train_items), len(LAB_ITEMS))
Y_LAB     = torch.tensor([float(l) for _, l in LAB_ITEMS], device=device)

for name, it in [("TRAIN", train_items), ("VAL", val_items)]:
    labs = [l for _, l in it]
    print(f"{name:5s}: {len(it):5d} videos | anomaly {sum(labs):4d} | normal {len(labs)-sum(labs):4d}")
print(f"TEST : {len(test_items):5d} videos | unlabelled (predictions only)")

## 4. Feature extraction — frozen R(2+1)D-18, every frame, cached to disk

- Same video input as notebook 02: every frame, consecutive non-overlapping 16-frame clips (the last clip is padded by
  repeating the last frame), resize to 128×171, centre crop 112×112, Kinetics normalisation.
- The backbone is the Kinetics-400 R(2+1)D-18 with its classifier removed → one 512-d feature per clip. It is **not**
  fine-tuned, so the features contain no information about our labels and can be reused for every fold of Part B.
- Videos are streamed in segments of `CLIPS_PER_SEGMENT` clips (the longest video has ~628k frames). The clip features of
  a video are concatenated in time order and saved as `FEATS_DIR/<view>/<category>/<video>.npy` (float16).
- Extraction is **resumable**: videos whose `.npy` already exists are skipped, so this cell is fast after the first run.

In [ ]:
def open_video(path):
    # 1 decode thread per video: the default (16) costs ~5x more RAM per open video and,
    # summed over all workers, oversubscribes the CPU.
    return cv2.VideoCapture(path, cv2.CAP_FFMPEG, [cv2.CAP_PROP_N_THREADS, 1])


def frame_count(path):
    cap = open_video(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


def video_fps(path):
    cap = open_video(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps and fps > 0 else 30.0


SEG_FRAMES = CLIP_LEN * CLIPS_PER_SEGMENT
CROP_Y, CROP_X = (RESIZE_HW[0] - CROP) // 2, (RESIZE_HW[1] - CROP) // 2


def read_frames(path, start, count):
    """Decodes `count` consecutive frames from frame `start` (count=None: up to the end of the video).
       Every frame is resized to RESIZE_HW and centre-cropped. Returns a list of uint8 (CROP, CROP, 3) RGB arrays."""
    cap = open_video(path)
    if start > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    def grab():
        frames = []
        while count is None or len(frames) < count:
            ok, fr = cap.read()
            if not ok:
                break
            fr = cv2.resize(fr, (RESIZE_HW[1], RESIZE_HW[0]), interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(fr[CROP_Y:CROP_Y + CROP, CROP_X:CROP_X + CROP], cv2.COLOR_BGR2RGB))
        return frames

    frames = grab()
    if not frames and start > 0:            # seek failed -> reopen and step through the frames before `start`
        cap.release(); cap = open_video(path)
        for _ in range(start):
            if not cap.grab():
                break
        frames = grab()
    cap.release()
    return frames


class SegmentDataset(Dataset):
    """One item = one segment of one video: up to CLIPS_PER_SEGMENT consecutive 16-frame clips, (N, C, T, H, W) uint8.
       The segments of a video together contain every one of its frames. Keys (vid, seg, flip) come from SegmentSampler."""
    def __init__(self, items):
        self.items = items
        self.n_frames = [frame_count(p) for p, _ in items]
        self.n_segs = [max(1, -(-n // SEG_FRAMES)) for n in self.n_frames]

    def __len__(self):
        return sum(self.n_segs)

    def __getitem__(self, key):
        vid, seg, flip = key
        path = self.items[vid][0]
        last = seg == self.n_segs[vid] - 1
        frames = read_frames(path, seg * SEG_FRAMES, None if last else SEG_FRAMES)   # last segment: read to the end
        if not frames and seg == 0:
            print(f"WARNING: could not decode {path}")
            frames = [np.zeros((CROP, CROP, 3), np.uint8)]
        if frames:
            frames += [frames[-1]] * (-len(frames) % CLIP_LEN)    # pad the last clip by repeating the last frame
        n_clips = len(frames) // CLIP_LEN
        clips = np.stack(frames) if frames else np.zeros((0, CROP, CROP, 3), np.uint8)
        if flip:
            clips = np.ascontiguousarray(clips[:, :, ::-1])
        clips = torch.from_numpy(clips).view(n_clips, CLIP_LEN, CROP, CROP, 3).permute(0, 4, 1, 2, 3)   # (N, C, T, H, W)
        return clips, vid, last


class SegmentSampler(Sampler):
    """Yields the segments of video 0, then video 1, ... in order (the DataLoader keeps this order with many workers)."""
    def __init__(self, ds, flip):
        self.ds, self.flip = ds, flip

    def __len__(self):
        return sum(self.ds.n_segs)

    def __iter__(self):
        for v in range(len(self.ds.items)):
            for s in range(self.ds.n_segs[v]):
                yield (v, s, self.flip)


def worker_init(_):
    cv2.setNumThreads(1)                     # one decode thread per worker; parallelism comes from workers


MEAN = torch.tensor([0.43216, 0.394666, 0.37645], device=device).view(1, 3, 1, 1, 1)   # Kinetics stats
STD  = torch.tensor([0.22803, 0.22145, 0.216989], device=device).view(1, 3, 1, 1, 1)

def to_input(x):
    x = x.to(device, non_blocking=True).float().div_(255)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last_3d)


def feat_path(path, view):
    rel = os.path.relpath(path, VIDEOS_DIR)
    return os.path.join(FEATS_DIR, view, os.path.splitext(rel)[0] + ".npy")


@torch.no_grad()
def extract_features(backbone, items, view, flip):
    """Saves the (N_clips, 512) float16 clip features of every video in `items` that is not cached yet."""
    todo = [it for it in items if not os.path.exists(feat_path(it[0], view))]
    print(f"[{view}] {len(items) - len(todo)} cached, {len(todo)} to extract")
    if not todo:
        return
    ds = SegmentDataset(todo)
    loader = DataLoader(ds, sampler=SegmentSampler(ds, flip), batch_size=None, num_workers=NUM_WORKERS,
                        pin_memory=True, worker_init_fn=worker_init,
                        prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None)
    chunks = []
    for clips, vid, last in tqdm(loader, leave=False, desc=f"extract ({view})"):
        if len(clips):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                chunks.append(backbone(to_input(clips)).float().cpu())
        if last:
            out = feat_path(todo[vid][0], view)
            os.makedirs(os.path.dirname(out), exist_ok=True)
            np.save(out[:-4] + ".tmp.npy", torch.cat(chunks).half().numpy())
            os.replace(out[:-4] + ".tmp.npy", out)                # atomic: an interrupted run never leaves a broken file
            chunks = []


backbone = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
backbone.fc = nn.Identity()                                  # stem -> layer1..4 -> avgpool -> 512-d clip feature
backbone = backbone.to(device).to(memory_format=torch.channels_last_3d).eval()

t0 = time.time()
for view, flip in VIEWS.items():
    extract_features(backbone, LAB_ITEMS + test_items, view, flip)
print(f"feature extraction done in {(time.time() - t0) / 60:.1f} min")
del backbone
torch.cuda.empty_cache()

### Load the cached features onto the GPU (float16, ~0.2 GB per view)

In [ ]:
def load_feats(items):
    """{view: [ (N_clips, 512) float16 tensor on the GPU, one per video ]}"""
    return {view: [torch.from_numpy(np.load(feat_path(p, view))).to(device) for p, _ in items] for view in VIEWS}


F_LAB  = load_feats(LAB_ITEMS)
F_TEST = load_feats(test_items)

for name, feats in [("TRAIN", [F_LAB["center"][i] for i in TRAIN_IDX]), ("VAL", [F_LAB["center"][i] for i in VAL_IDX]),
                    ("TEST", F_TEST["center"])]:
    n = np.array([len(f) for f in feats])
    print(f"{name:5s}: {n.sum():6d} clips | clips/video min {n.min()} median {int(np.median(n))} max {n.max()}")

## 5. Model: gated attention MIL over the clip features of a video

For a video with clip features $x_1 \dots x_N$:
1. **Temporal conv** (kernel 3): $h_i$ mixes each clip with its neighbours. An event spans several 16-frame clips.
2. **Gated attention**: $a_i = \mathrm{softmax}_i\big(w^\top(\tanh(V h_i) \odot \sigma(U h_i))\big)$, with $\sum_i a_i = 1$.
   This learns how much each clip should count for the whole video.
3. **Video logit** = $\mathrm{classifier}\big(\sum_i a_i h_i\big)$.

The classifier is linear, so the video logit equals $\sum_i a_i \cdot \mathrm{classifier}(h_i)$: an attention-weighted
average of **per-clip anomaly logits**. Mean pooling (notebook 02) is the special case $a_i = 1/N$. Here the model can put
almost all the weight on the few clips that contain the event. The loss is still one BCE per video, because only
video labels exist.

In [ ]:
class AttentionMIL(nn.Module):
    """Gated attention MIL (Ilse et al., 2018) with a temporal conv in front.
       Input: (N_clips, 512) features of one video.
       Output: video logit (1,), attention weights (N_clips,) summing to 1, per-clip anomaly logits (N_clips,)."""
    def __init__(self, d_in=512, hidden=HIDDEN, attn_dim=ATTN_DIM, p_drop=DROPOUT):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.embed = nn.Sequential(nn.Conv1d(d_in, hidden, kernel_size=3, padding=1), nn.GELU(), nn.Dropout(p_drop))
        self.attn_v = nn.Linear(hidden, attn_dim)
        self.attn_u = nn.Linear(hidden, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.embed(self.norm(x.float()).T[None])[0].T                    # (N, hidden)
        a = self.attn_w(torch.tanh(self.attn_v(h)) * torch.sigmoid(self.attn_u(h))).squeeze(1)
        a = torch.softmax(a, dim=0)                                         # (N,)
        return self.classifier(a @ h), a, self.classifier(h).squeeze(1)


def make_criterion(idx):
    """BCE with pos_weight = #normal / #anomaly of the training videos `idx` (same balancing as notebook 02)."""
    n_pos = Y_LAB[idx].sum()
    return nn.BCEWithLogitsLoss(pos_weight=((len(idx) - n_pos) / n_pos).view(1))


def video_logits(model, feats, idx):
    """One logit per video: the model's video logit averaged over all views (test-time augmentation)."""
    return torch.cat([torch.stack([model(feats[v][i])[0] for v in VIEWS]).mean(0) for i in idx])


def train_epoch(model, optimizer, crit, idx):
    """One pass over the training videos in random order, one random view per video, BATCH_VIDEOS videos per step."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    order = np.random.permutation(idx)
    losses, correct = [], 0
    for n, i in enumerate(order, 1):
        logit = model(F_LAB[random.choice(list(VIEWS))][i])[0]
        loss = crit(logit, Y_LAB[i:i + 1])
        (loss / BATCH_VIDEOS).backward()
        losses.append(loss.item())
        correct += int((logit.item() > 0) == bool(Y_LAB[i]))
        if n % BATCH_VIDEOS == 0 or n == len(order):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    return float(np.mean(losses)), correct / len(order)


@torch.no_grad()
def evaluate(model, crit, idx):
    """loss, accuracy, ROC-AUC and logits on the labelled videos `idx` (all views averaged, no dropout)."""
    model.eval()
    logits = video_logits(model, F_LAB, idx)
    y = Y_LAB[idx]
    acc = ((logits > 0).float() == y).float().mean().item()
    return crit(logits, y).item(), acc, roc_auc_score(y.cpu().numpy(), logits.cpu().numpy()), logits.cpu().numpy()


def fit(train_idx, val_idx, epochs=EPOCHS, select_best=False, seed=SEED, verbose=True):
    """Trains a new AttentionMIL head on `train_idx`, evaluating on `val_idx` after every epoch.
       select_best=True: returns the weights of the epoch with the lowest val loss (smoother than accuracy on 53 videos).
       select_best=False: returns the last epoch (used for cross-validation, where no fold looks at its own val set)."""
    seed_everything(seed)
    model = AttentionMIL().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    crit = make_criterion(train_idx)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_auc"]}
    best_loss, best_state, best_ep = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, optimizer, crit, train_idx)
        scheduler.step()
        va_loss, va_acc, va_auc, _ = evaluate(model, crit, val_idx)
        for k, v in zip(hist, [tr_loss, tr_acc, va_loss, va_acc, va_auc]):
            hist[k].append(v)
        flag = ""
        if select_best and va_loss < best_loss:
            best_loss, best_state, best_ep = va_loss, copy.deepcopy(model.state_dict()), ep
            flag = "  <- best"
        if verbose:
            print(f"epoch {ep:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} auc {va_auc:.4f} | {time.time()-t0:.1f}s{flag}")
    if select_best:
        model.load_state_dict(best_state)
        if verbose:
            print(f"Loaded best epoch {best_ep} (val loss {best_loss:.4f})")
    model.eval()
    return model, hist, best_ep


print("trainable params:", sum(p.numel() for p in AttentionMIL().parameters()))

## 6. Plotting and metric helpers

In [ ]:
def plot_history(hist, title, best_ep=None, save_name=None):
    """hist values may be lists (one run) or arrays of shape (folds, epochs) (mean ± std is drawn)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, key, name in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy"), (axes[2], "auc", "ROC-AUC")]:
        for split, marker in [("train", "o-"), ("val", "s-")]:
            if f"{split}_{key}" not in hist:
                continue
            v = np.atleast_2d(np.array(hist[f"{split}_{key}"]))
            ep = np.arange(1, v.shape[1] + 1)
            m, s = v.mean(0), v.std(0)
            ax.plot(ep, m, marker, ms=3, label=f"{split.capitalize()} {name.lower()}")
            if len(v) > 1:
                ax.fill_between(ep, m - s, m + s, alpha=0.2)
        if best_ep:
            ax.axvline(best_ep, color="red", ls="--", lw=1.5, label=f"Selected epoch ({best_ep})")
        ax.set_xlabel("Epoch"); ax.set_ylabel(name); ax.set_title(f"{title} — {name}")
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(RUNS_DIR, save_name), dpi=150)
    plt.show()


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, np.float64)))


def metric_dict(y, probs):
    pred = (probs >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, probs) if len(set(y)) == 2 else float("nan")}


def report(probs, items, split_name, stage_name):
    y = np.array([l for _, l in items])
    pred = (probs >= 0.5).astype(int)
    m = metric_dict(y, probs)
    print(f"===== {stage_name} | {split_name} ({len(items)} videos, every frame) =====")
    print(pd.DataFrame([m], index=[split_name]).round(4).to_string())
    print(classification_report(y, pred, target_names=CLASSES, digits=4, zero_division=0))

    df = pd.DataFrame({"video": [os.path.relpath(p, VIDEOS_DIR) for p, _ in items],
                       "category": [category(p) for p, _ in items],
                       "label": np.array(CLASSES)[y], "anomaly_prob": probs.round(4),
                       "prediction": np.array(CLASSES)[pred]})
    df["correct"] = df["label"] == df["prediction"]
    print("Per-category accuracy:")
    print(df.groupby("category")["correct"].agg(["mean", "size"]).rename(columns={"mean": "acc", "size": "videos"})
            .round(3).to_string(), "\n")

    cm = confusion_matrix(y, pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{stage_name} — {split_name}")
    plt.tight_layout(); plt.show()
    return m


@torch.no_grad()
def predict_test(models, stage_name, file_tag):
    """Anomaly probability of every TEST video: sigmoid of the video logit averaged over views and over `models`."""
    logits = np.mean([video_logits(m.eval(), F_TEST, range(len(test_items))).cpu().numpy() for m in models], axis=0)
    probs = sigmoid(logits)
    df = pd.DataFrame({"video": [os.path.relpath(p, VIDEOS_DIR) for p, _ in test_items],
                       "anomaly_prob": probs.round(4),
                       "prediction": np.where(probs >= 0.5, CLASSES[1], CLASSES[0])})
    out = os.path.join(RUNS_DIR, f"test_predictions_{file_tag}.csv")
    df.to_csv(out, index=False)
    print(f"===== {stage_name} | TEST ({len(df)} videos) -> {out} =====")
    print(df["prediction"].value_counts().to_string())
    return df

## 7. Part A — Attention MIL on the same TRAIN / VAL split as notebook 02

Trains in seconds per epoch because the backbone features are cached. The epoch with the lowest VAL loss is kept
(notebook 02 kept the epoch with the highest VAL accuracy).

In [ ]:
model_a, hist_a, best_ep_a = fit(TRAIN_IDX, VAL_IDX, select_best=True)
torch.save({"model": model_a.state_dict(), "epoch": best_ep_a}, os.path.join(RUNS_DIR, "attnmil_best.pt"))

### Part A — accuracy, loss & ROC-AUC curves

In [ ]:
plot_history(hist_a, "Attention MIL (Part A)", best_ep=best_ep_a, save_name="curves_part_a.png")

### Part A — accuracy, precision, recall, F1, ROC-AUC (VAL) + TEST predictions

In [ ]:
metrics = {}
_, _, _, val_logits_a = evaluate(model_a, make_criterion(TRAIN_IDX), VAL_IDX)
metrics[("Attention MIL", "VAL")] = report(sigmoid(val_logits_a), val_items, "VAL", "Attention MIL")
test_pred_a = predict_test([model_a], "Attention MIL", "attnmil")

## 8. Anomaly timelines — where in the video does the model look?

For some VAL videos: the attention weight of every clip (how much it decides the video, scaled so the largest is 1) and
the clip's own anomaly probability, against time in the video. Anomaly videos should show a few sharp attention peaks at
the event. Normal videos should show low clip probabilities throughout.

The model was never told where the events are. It only received one label per video.

In [ ]:
@torch.no_grad()
def plot_timelines(model, idx_list, save_name=None):
    fig, axes = plt.subplots(len(idx_list), 1, figsize=(14, 2.6 * len(idx_list)), squeeze=False)
    for ax, i in zip(axes[:, 0], idx_list):
        path, label = LAB_ITEMS[i]
        logit, attn, clip_logits = model(F_LAB["center"][i])
        attn = attn.cpu().numpy(); attn = attn / attn.max()
        t = (np.arange(len(attn)) * CLIP_LEN + CLIP_LEN / 2) / video_fps(path)
        unit = "min" if t[-1] > 600 else "s"
        t = t / 60 if unit == "min" else t
        ax.fill_between(t, 0, attn, color="tab:orange", alpha=0.5, label="attention (max = 1)")
        ax.plot(t, sigmoid(clip_logits.cpu().numpy()), color="tab:blue", lw=1, label="clip anomaly prob")
        ax.set_ylim(0, 1.05); ax.set_xlim(t[0], t[-1]); ax.set_xlabel(f"time ({unit})")
        ax.set_title(f"{os.path.relpath(path, VIDEOS_DIR)} | true {CLASSES[label]} | "
                     f"video anomaly prob {sigmoid(logit.item()):.3f}", fontsize=10)
        ax.grid(alpha=0.3); ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(RUNS_DIR, save_name), dpi=120)
    plt.show()


# 4 VAL anomaly videos (highest predicted probability) + 2 VAL normal videos
order = VAL_IDX[np.argsort(-val_logits_a)]
show = [i for i in order if Y_LAB[i] == 1][:4] + [i for i in order[::-1] if Y_LAB[i] == 0][:2]
plot_timelines(model_a, show, save_name="timelines_val.png")

## 9. Part B — 5-fold cross-validation over all labelled videos (TRAIN + VAL = 350 videos)

Every labelled video is in the validation fold exactly once, so the **out-of-fold** accuracy is measured on 350 videos
instead of 53. Folds are stratified by category. Each fold trains for a fixed `EPOCHS` and uses the **last** epoch, so no fold
picks its checkpoint by looking at its own validation videos. This is the most honest number in the notebook.

The 5 fold models are also averaged into an ensemble for the TEST predictions.

In [ ]:
cats = [category(p) for p, _ in LAB_ITEMS]
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The least populated class")   # e.g. Arson has 4 videos < 5 folds
    folds = list(skf.split(np.zeros(len(LAB_ITEMS)), cats))

oof_logits = np.zeros(len(LAB_ITEMS))
fold_models, fold_hists, fold_rows = [], [], []
for k, (tr, va) in enumerate(folds, 1):
    t0 = time.time()
    model_k, hist_k, _ = fit(tr, va, select_best=False, seed=SEED + k, verbose=False)
    _, _, _, oof_logits[va] = evaluate(model_k, make_criterion(tr), va)
    y_va = Y_LAB[va].cpu().numpy().astype(int)
    fold_rows.append(metric_dict(y_va, sigmoid(oof_logits[va])))
    fold_models.append(model_k); fold_hists.append(hist_k)
    torch.save(model_k.state_dict(), os.path.join(RUNS_DIR, f"cv_fold{k}.pt"))
    print(f"fold {k}/{N_FOLDS} | train {len(tr)} val {len(va)} | val acc {fold_rows[-1]['accuracy']:.4f} "
          f"auc {fold_rows[-1]['roc_auc']:.4f} | {time.time()-t0:.0f}s")

fold_df = pd.DataFrame(fold_rows, index=[f"fold {k}" for k in range(1, N_FOLDS + 1)])
fold_df.loc["mean"] = fold_df.mean(); fold_df.loc["std"] = fold_df.iloc[:N_FOLDS].std()
fold_df.round(4)

### Part B — curves averaged over the 5 folds (shaded = ± 1 std)

In [ ]:
plot_history({k: [h[k] for h in fold_hists] for k in fold_hists[0]}, f"Attention MIL ({N_FOLDS}-fold CV)",
             save_name="curves_cv.png")

### Part B — out-of-fold metrics on all 350 labelled videos + TEST predictions (5-fold ensemble)

In [ ]:
metrics[(f"Attention MIL, {N_FOLDS}-fold CV (out-of-fold)", "TRAIN+VAL")] = report(
    sigmoid(oof_logits), LAB_ITEMS, "TRAIN+VAL (out-of-fold)", f"Attention MIL {N_FOLDS}-fold CV")
test_pred_cv = predict_test(fold_models, f"Attention MIL {N_FOLDS}-fold ensemble", "attnmil_cv_ensemble")

## 10. Comparison with notebook 02 (mean pooling over the whole video)

The VAL rows use the same 53 videos. The cross-validation row uses all 350 labelled videos, so it is not directly
comparable to the VAL rows, but it is the most reliable estimate of accuracy.

In [ ]:
summary = pd.DataFrame({k: v for k, v in metrics.items() if v is not None}).T
summary.index.names = ["stage", "split"]
summary.to_csv(os.path.join(RUNS_DIR, "metrics_summary.csv"))

prev_csv = os.path.join(PREV_RUNS_DIR, "metrics_summary.csv")
if os.path.exists(prev_csv):
    prev = pd.read_csv(prev_csv).set_index(["stage", "split"])
    prev.index = pd.MultiIndex.from_tuples([(f"02 mean pooling: {s}", sp) for s, sp in prev.index], names=["stage", "split"])
    summary = pd.concat([prev, summary])
summary.round(4)